In [9]:
with open("/content/chat.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()

print(lines[:10])

['Messages and calls are end-to-end encrypted. No one outside of this chat, not even WhatsApp, can read or listen to them.\n', '[5/5/26, 8:01:13 PM] - [Call]\n', '[5/21/26, 8:01:15 AM] - Sarii added You\n', '[5/21/26, 2:00:01 PM] You: [Forwarded] <document omitted> manu7.pdf\n', '[5/21/26, 2:00:01 PM] You: [Forwarded] <document omitted> dbms lab 03.pdf\n', '[5/21/26, 7:06:18 PM] Sarii: [Forwarded] <album message>\n', '[5/21/26, 7:06:18 PM] Sarii: [Forwarded] <image omitted>\n', '[5/21/26, 7:06:18 PM] Sarii: [Forwarded] <image omitted>\n', '[5/21/26, 7:06:19 PM] Sarii: [Forwarded] <image omitted>\n', '[5/21/26, 7:06:19 PM] Sarii: <unknown message>\n']


In [11]:
# ==========================================
# WHATSAPP GROUPDNA ANALYZER
# NO IMPORTS USED - ONLY BASIC PYTHON
# ==========================================
#
# Works with the WhatsApp export format that looks like:
#   [5/21/26, 2:00:01 PM] Sarii: message text here
# Multi-line messages (a message that continues on the next
# line without a new "[date, time]" tag) are stitched back
# together automatically. System lines such as
#   [5/5/26, 8:01:13 PM] - [Call]
#   [5/21/26, 8:01:15 AM] - Sarii added You
# are recognised and skipped from the stats.


# ---------- SETTINGS ----------

CHAT_FILE = "chat.txt"              # exported WhatsApp .txt file
GROUP_NAME = "mad people's"    # change this to your group's name

# Words to ignore when building the "favourite words" list
stop_words = [
    "the", "and", "for", "you", "are", "was", "not", "but", "with",
    "hai", "hua", "nahi", "kar", "kara", "karo", "hoga", "hi", "ho",
    "bol", "ko", "ka", "ki", "ke", "se", "toh", "to", "aur", "abi",
    "abhi", "yeh", "wo", "woh", "tha", "this", "that", "message"
]

# Media / system snippets to exclude from word counts
ignore_snippets = [
    "<image omitted>", "<video omitted>", "<audio omitted>",
    "<document omitted>", "<sticker omitted>", "<gif omitted>",
    "<media omitted>", "<unknown message>", "<album message>",
    "you deleted this message", "this message was deleted",
    "[forwarded]", "missed voice call", "missed video call"
]

messages = []

people = {}
hour_count = {}
day_count = {}
word_count = {}

response_times = {}
last_message_time = {}

# 24 hours
for i in range(24):
    hour_count[i] = 0


# ==========================================
# DATE AND TIME FUNCTIONS
# ==========================================

def days_in_month(month, year):

    if month == 2:
        if year % 4 == 0:
            return 29
        else:
            return 28

    if month == 4 or month == 6 or month == 9 or month == 11:
        return 30

    return 31


def date_to_number(day, month, year):

    total = 0
    y = 2000

    while y < year:
        if y % 4 == 0:
            total = total + 366
        else:
            total = total + 365
        y = y + 1

    m = 1
    while m < month:
        total = total + days_in_month(m, year)
        m = m + 1

    total = total + day

    return total


def time_to_minutes(day, month, year, hour, minute):
    date_number = date_to_number(day, month, year)
    return date_number * 1440 + hour * 60 + minute


# ==========================================
# HEADER PARSER (replaces regex)
# Tries to read "[M/D/YY, H:MM:SS AM/PM] rest..."
# Returns None if the line is not a valid header line
# (in which case it's treated as a continuation line).
# ==========================================

def parse_header(line):

    if len(line) == 0 or line[0] != "[":
        return None

    close_bracket = line.find("]")
    if close_bracket == -1:
        return None

    header = line[1:close_bracket]

    comma_position = header.find(",")
    if comma_position == -1:
        return None

    date_text = header[:comma_position].strip()
    time_text = header[comma_position + 1:].strip()

    # ---- date ----
    date_values = date_text.split("/")
    if len(date_values) != 3:
        return None

    try:
        month = int(date_values[0])
        day = int(date_values[1])
        year = int(date_values[2])
    except:
        return None

    if year < 100:
        year = 2000 + year

    # ---- time ----
    time_text_clean = time_text.lower()
    is_pm = "pm" in time_text_clean
    is_am = "am" in time_text_clean

    time_text_clean = time_text_clean.replace("am", "")
    time_text_clean = time_text_clean.replace("pm", "")
    time_text_clean = time_text_clean.strip()

    time_values = time_text_clean.split(":")
    if len(time_values) < 2:
        return None

    try:
        hour = int(time_values[0])
        minute = int(time_values[1])
    except:
        return None

    if is_pm and hour != 12:
        hour = hour + 12
    if is_am and hour == 12:
        hour = 0

    # ---- rest of line (after "] ") ----
    if close_bracket + 2 <= len(line):
        rest = line[close_bracket + 2:]
    else:
        rest = ""

    return [day, month, year, hour, minute, rest]


# ==========================================
# READ FILE
# ==========================================

lines = []

try:
    file = open(CHAT_FILE, "r", encoding="utf-8")
    lines = file.readlines()
    file.close()
except:
    print("Could not find '" + CHAT_FILE + "'. Place your exported chat file next to this script.")
    lines = []


# ==========================================
# PARSE WHATSAPP MESSAGES (handles multi-line messages)
# ==========================================

current = None   # message currently being built

for raw_line in lines:

    line = raw_line.rstrip("\n")
    line = line.rstrip("\r")

    parsed = parse_header(line)

    if parsed is not None:

        # a new timestamped entry starts -> close off the previous one
        if current is not None:
            messages.append(current)
            current = None

        day, month, year, hour, minute, rest = parsed

        # System / notification lines look like "- [Call]" or
        # "- Sarii added You" -> no real sender, skip from stats.
        if rest[:2] == "- ":
            continue

        colon_position = rest.find(": ")
        if colon_position == -1:
            continue

        person = rest[:colon_position].strip()
        text = rest[colon_position + 2:]

        if person == "":
            continue

        current = {
            "person": person,
            "text": text,
            "day": day,
            "month": month,
            "year": year,
            "hour": hour,
            "minute": minute
        }

    else:
        # continuation of a multi-line message
        if current is not None and line.strip() != "":
            current["text"] = current["text"] + "\n" + line

if current is not None:
    messages.append(current)


# ==========================================
# STOP IF NO MESSAGES FOUND
# ==========================================

if len(messages) == 0:
    print("No valid WhatsApp messages found in '" + CHAT_FILE + "'.")
    print("Make sure the file is a WhatsApp exported chat .txt file.")


# ==========================================
# MESSAGE COUNT PER PERSON
# ==========================================

for message in messages:
    person = message["person"]
    if person not in people:
        people[person] = 0
    people[person] = people[person] + 1


# ==========================================
# HOUR ANALYSIS
# ==========================================

for message in messages:
    hour = message["hour"]
    hour_count[hour] = hour_count[hour] + 1


# ==========================================
# DAY ANALYSIS
# ==========================================

for message in messages:
    d = (
        str(message["day"]) + "/" +
        str(message["month"]) + "/" +
        str(message["year"])
    )
    if d not in day_count:
        day_count[d] = 0
    day_count[d] = day_count[d] + 1


# ==========================================
# FAVOURITE WORDS
# ==========================================

punctuation = [
    ".", ",", "!", "?", ":", ";",
    "(", ")", "[", "]", "{", "}",
    "'", '"', "-", "_", "/", "\\",
    "@", "#", "$", "%", "&", "*", "<", ">"
]

for message in messages:

    text = message["text"].lower()

    skip = False
    for snippet in ignore_snippets:
        if snippet in text:
            skip = True
            break
    if skip:
        continue

    for symbol in punctuation:
        text = text.replace(symbol, " ")

    words = text.split()

    for word in words:

        if len(word) <= 2:
            continue
        if word in stop_words:
            continue
        if word.isdigit():
            continue

        if word not in word_count:
            word_count[word] = 0
        word_count[word] = word_count[word] + 1


# ==========================================
# SORTING FUNCTION (bubble sort, highest first)
# ==========================================

def sort_dictionary(dictionary):

    items = []
    for key in dictionary:
        items.append([key, dictionary[key]])

    n = len(items)
    i = 0

    while i < n:
        j = 0
        while j < n - i - 1:
            if items[j][1] < items[j + 1][1]:
                temp = items[j]
                items[j] = items[j + 1]
                items[j + 1] = temp
            j = j + 1
        i = i + 1

    return items


# ==========================================
# BUSIEST DAY / HOUR
# ==========================================

sorted_days = sort_dictionary(day_count)
busiest_day = sorted_days[0] if len(sorted_days) > 0 else ["None", 0]

sorted_hours = sort_dictionary(hour_count)
busiest_hour = sorted_hours[0] if len(sorted_hours) > 0 else [0, 0]


# ==========================================
# ACTIVITY HEATMAP DATA (per person, per hour)
# ==========================================

person_hours = {}
for person in people:
    person_hours[person] = {}
    for hour in range(24):
        person_hours[person][hour] = 0

for message in messages:
    person = message["person"]
    hour = message["hour"]
    person_hours[person][hour] = person_hours[person][hour] + 1


# ==========================================
# RESPONSE TIME
# ==========================================

for message in messages:

    person = message["person"]

    current_time = time_to_minutes(
        message["day"],
        message["month"],
        message["year"],
        message["hour"],
        message["minute"]
    )

    if person not in response_times:
        response_times[person] = []

    if person in last_message_time:
        difference = current_time - last_message_time[person]
        # Ignore negative gaps and gaps larger than a day
        if 0 <= difference <= 1440:
            response_times[person].append(difference)

    last_message_time[person] = current_time


# ==========================================
# AVERAGE RESPONSE TIME
# ==========================================

average_response = {}

for person in response_times:
    times = response_times[person]
    if len(times) > 0:
        total = 0
        for value in times:
            total = total + value
        average_response[person] = total / len(times)
    else:
        average_response[person] = 0


# ==========================================
# FASTEST / SLOWEST REPLIER
# ==========================================

fastest_person = ""
fastest_time = 999999

slowest_person = ""
slowest_time = 0

for person in average_response:
    value = average_response[person]
    if value > 0:
        if value < fastest_time:
            fastest_time = value
            fastest_person = person
        if value > slowest_time:
            slowest_time = value
            slowest_person = person


# ==========================================
# SILENT STREAKS
# ==========================================

person_dates = {}

for message in messages:
    person = message["person"]
    date_number = date_to_number(message["day"], message["month"], message["year"])
    if person not in person_dates:
        person_dates[person] = []
    person_dates[person].append(date_number)

silent_streaks = {}

for person in person_dates:

    dates = person_dates[person]

    unique_dates = []
    for d in dates:
        if d not in unique_dates:
            unique_dates.append(d)

    unique_dates.sort()

    longest_gap = 0
    i = 1
    while i < len(unique_dates):
        gap = unique_dates[i] - unique_dates[i - 1]
        if gap > longest_gap:
            longest_gap = gap
        i = i + 1

    silent_streaks[person] = longest_gap


# ==========================================
# NUMBER OF DAYS THE CHAT SPANS
# ==========================================

first_date_number = 0
last_date_number = 0

i = 0
while i < len(messages):

    date_number = date_to_number(
        messages[i]["day"],
        messages[i]["month"],
        messages[i]["year"]
    )

    if i == 0:
        first_date_number = date_number
        last_date_number = date_number
    else:
        if date_number < first_date_number:
            first_date_number = date_number
        if date_number > last_date_number:
            last_date_number = date_number

    i = i + 1

total_days = (last_date_number - first_date_number) + 1 if len(messages) > 0 else 0


# ==========================================
# ADD COMMAS TO A NUMBER (e.g. 3174 -> 3,174)
# ==========================================

def add_commas(number):

    number_text = str(number)
    result = ""
    count = 0
    i = len(number_text) - 1

    while i >= 0:
        result = number_text[i] + result
        count = count + 1
        if count % 3 == 0 and i != 0:
            result = "," + result
        i = i - 1

    return result


# ==========================================
# PRINT REPORT
# ==========================================

message_total_text = add_commas(len(messages))

subheading = (
    str(total_days) + " days  \u2022  " +
    message_total_text + " messages  \u2022  " +
    str(len(people)) + " members"
)

print()
print("=" * 60)
print('  GROUPDNA REPORT \u2014 "' + GROUP_NAME + '"')
print("  " + subheading)
print("=" * 60)
print()


# ---------- MESSAGES PER PERSON ----------

print("MESSAGES PER PERSON")
print("-" * 40)

sorted_people = sort_dictionary(people)

for item in sorted_people:

    person = item[0]
    count = item[1]

    percentage = (count / len(messages)) * 100 if len(messages) > 0 else 0
    bar_length = int(percentage / 2)
    bar = "*" * bar_length

    print(
        person.ljust(12),
        bar.ljust(30),
        count,
        "(" + str(round(percentage, 1)) + "%)"
    )


# ---------- BUSIEST DAY ----------

print()
print("BUSIEST DAY")
print("-" * 40)
print(busiest_day[0], "(", busiest_day[1], "messages)")


# ---------- BUSIEST HOUR ----------

print()
print("BUSIEST HOUR")
print("-" * 40)
print(
    str(busiest_hour[0]).zfill(2) + ":00 - " +
    str((busiest_hour[0] + 1) % 24).zfill(2) + ":00",
    "(", busiest_hour[1], "messages)"
)


# ---------- ACTIVITY HEATMAP ----------

print()
print("ACTIVITY HEATMAP")
print("(hours 00 to 23)")
print("-" * 40)

print("            ", end="")
for hour in range(24):
    print(str(hour).zfill(2), end=" ")
print()

for person in people:

    print(person[:10].ljust(12), end="")

    for hour in range(24):
        value = person_hours[person][hour]

        if value == 0:
            symbol = "."
        elif value <= 2:
            symbol = "o"
        elif value <= 5:
            symbol = "O"
        elif value <= 10:
            symbol = "*"
        else:
            symbol = "_"

        print(symbol.ljust(2), end=" ")

    print()


# ---------- FAVOURITE WORDS ----------

print()
print("THIS GROUP'S FAVOURITE WORDS")
print("-" * 40)

sorted_words = sort_dictionary(word_count)

counter = 0
for item in sorted_words:
    word = item[0]
    count = item[1]
    print(word.ljust(15), count)
    counter = counter + 1
    if counter == 10:
        break


# ---------- RESPONSE PATTERNS ----------

print()
print("RESPONSE PATTERNS")
print("-" * 40)

if fastest_person != "":
    print("Fastest replier :", fastest_person, "(", round(fastest_time, 1), "minutes)")
else:
    print("Fastest replier : Not enough data")

if slowest_person != "":
    print("Slowest replier :", slowest_person, "(", round(slowest_time / 60, 1), "hours)")
else:
    print("Slowest replier : Not enough data")


# ---------- AVERAGE RESPONSE TIME ----------

print()
print("AVERAGE RESPONSE TIME")
print("-" * 40)

for person in average_response:
    value = average_response[person]
    if value > 0:
        if value < 60:
            print(person, ":", round(value, 1), "minutes")
        else:
            print(person, ":", round(value / 60, 1), "hours")


# ---------- SILENT STREAKS ----------

print()
print("LONGEST SILENT STREAKS")
print("-" * 40)

sorted_silent = sort_dictionary(silent_streaks)

for item in sorted_silent:
    person = item[0]
    gap_days = item[1]
    print(person, ":", gap_days, "days")


# ---------- SUMMARY ----------

print()
print("=" * 60)
print("                  SUMMARY")
print("=" * 60)
print()

if len(sorted_people) > 0:
    print("Most active member :", sorted_people[0][0])
else:
    print("Most active member : None")

print("Busiest day        :", busiest_day[0])
print("Busiest hour       :", str(busiest_hour[0]).zfill(2) + ":00")

if len(sorted_words) > 0:
    print("Most used word      :", sorted_words[0][0])
else:
    print("Most used word      : None")

print()
print("=" * 60)


  GROUPDNA REPORT — "mad people's"
  90 days  •  554 messages  •  3 members

MESSAGES PER PERSON
----------------------------------------
Sadhana Kantakutle *******************            219 (39.5%)
You          *****************              192 (34.7%)
Sarii        ************                   143 (25.8%)

BUSIEST DAY
----------------------------------------
6/7/2026 ( 51 messages)

BUSIEST HOUR
----------------------------------------
21:00 - 22:00 ( 87 messages)

ACTIVITY HEATMAP
(hours 00 to 23)
----------------------------------------
            00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23 
You         #  o  .  .  .  .  o  .  o  @  O  @  #  @  o  O  O  .  #  #  #  @  O  @  
Sarii       o  .  .  .  o  .  O  o  .  @  #  @  O  #  .  #  O  @  #  #  #  @  O  #  
Sadhana Ka  @  o  .  .  .  .  .  O  o  #  @  O  @  @  .  .  O  #  @  @  @  @  #  @  

THIS GROUP'S FAVOURITE WORDS
----------------------------------------
node            19
return          16
